In [3]:
from transformers import BertForTokenClassification, AutoTokenizer, BertTokenizerFast
import torch
import numpy as np

In [4]:
tokenizer = BertTokenizerFast.from_pretrained("./../model/ner_tokenizer")
model = BertForTokenClassification.from_pretrained("./../model/ner_model")

In [6]:
# loading labels list

import pickle
with open('./../dataset/ner_label_list.pkl', 'rb') as f:
    label_list = pickle.load(f)

In [7]:
def make_prediction(text):
    model.eval()
    model.to("cuda")
    tokenized_text = tokenizer(text, return_tensors="pt", return_token_type_ids=False, is_split_into_words=False)
    output = model(**{k:v.to("cuda") for k,v in tokenized_text.items()})
    ner_indexes = output.logits.detach().cpu().numpy().argmax(axis=2)[0][1:-1]
    ner_texts = [label_list[i] for i in output.logits.detach().cpu().numpy().argmax(axis=2)[0]][1:-1]
    return ner_indexes.tolist(), ner_texts

In [8]:
tokenizer("শুক্রবার ভোর রাতে চিকিৎসাধীন অবস্থায় রাজধানীর স্কয়ার হাসপাতালে শেষ নিঃশ্বাস ত্যাগ করেন দ্বিজেন শর্মা", return_tensors="pt", return_token_type_ids=False, is_split_into_words=False).tokens()

['[CLS]',
 'শুক্রবার',
 'ভোর',
 'রাতে',
 'চিকিৎসাধীন',
 'অবস্থায়',
 'রাজধানীর',
 'স্কয়ার',
 'হাসপাতালে',
 'শেষ',
 'নিঃশ্বাস',
 'ত্যাগ',
 'করেন',
 'দ্বিজেন',
 'শর্মা',
 '[SEP]']

In [9]:
make_prediction("শুক্রবার ভোর রাতে চিকিৎসাধীন অবস্থায় রাজধানীর স্কয়ার হাসপাতালে শেষ নিঃশ্বাস ত্যাগ করেন দ্বিজেন শর্মা")

([0, 0, 0, 0, 0, 0, 3, 6, 0, 0, 0, 0, 1, 4],
 ['O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-ORG',
  'I-ORG',
  'O',
  'O',
  'O',
  'O',
  'B-PER',
  'I-PER'])